# NSBI diagnostics

In the previous notebook, we've trained CARL models that approximate the density ratio between two hypotheses of an event:

$$
 \hat{r}(x ;\mu_1,\mu_0) \approx \frac{\int dz\,p(x,z;\mu_1)}{\int dz\,p(x,z;\mu_0)}
$$

The validity of our NSBI results depends on close the “$\approx$” approximation holds. We will perform two diagnostics to gauge how well our estimates are performing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import vector
import hist

import torch
torch.set_float32_matmul_precision('medium')
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

from physics.analysis import zz4l, zz2l2v
from datasets.balanced import BalancedDataset
from nsbi import carl

## 0. Load the training/validation data, scaler, and model checkpoint

First, we'll load everything we need. (Note that by using the same seed for the two models, you only need to load denominator hypothesis dataset once)

In [ ]:
run_dir = '/global/cfs/cdirs/m5295/NSBIData/carl_models'

(events_sig_train, events_sig_val), (events_bkg_train, events_bkg_val) = carl.utils.load_data(run_dir, 'sig_over_bkg')
(events_sbi_train, events_sbi_val), _ = carl.utils.load_data(run_dir, 'sbi_over_bkg')  # same seed!

scaler_sig_over_bkg, model_sig_over_bkg = carl.utils.load_results(run_dir, 'sig_over_bkg')
scaler_sbi_over_bkg, model_sbi_over_bkg = carl.utils.load_results(run_dir, 'sbi_over_bkg')

features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 
               'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 
               'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 
               'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

## 1. Expectation value

A fundamental property that a probability distribution function must satisfy is that its integral is $1$. In other words, the expectation value of any density ratio estimate (DRE) over $\mu_0$ must follow

$$
\left< \hat r(x ; \mu_1, \mu_0) \right>_{\mu_0} = \int dx \; \frac{\hat p(x ;\mu_1)}{{\hat p(x;\mu_0)}} \; {p(x;\mu_0)} = 1.
$$

Let's explicitly verify this for each of the two DREs, 

$$
\frac{\hat p(x;\mu_{\rm S})}{\hat p(x;\mu_{\rm B})} \qquad \frac{\hat p(x;\mu_{\rm SBI})}{\hat p(x;\mu_{\rm B})}
$$

Using the training data we:

1. Scale the background events' features, $x$.
2. Obtain the classifier decisions, $\hat s(x)$, over the events.
3. Perform the likelihood trick over the output, $\hat r = \hat s/(1-\hat s)$.
4. Compute $C = \left< r \right>^{-1}_{\mu_{\rm B}}$.


In [ ]:
def calibration_factor(dre, scaler, events_denom):
    """
    Evaluate the calibration factor of a DRE, i.e. inverse of its expectation value over denominator hypothesis.
    
    Parameters
    ----------
    dre : torch.nn.Module
        Trained DRE model (e.g. a CARL instance loaded from a checkpoint).
    events_denom : array-like or torch.Tensor
        Input events sampled from the denominator hypothesis distribution.
    scaler : sklearn.preprocessing.StandardScaler or similar
        Fitted scaler used to normalize the input features before passing them to the model.
    
    Returns
    -------
    torch.Tensor
        Model predictions for the density ratio evaluated on the input events.
    """
    # compute denominator hypothesis density
    p_denom = torch.tensor(events_denom.weights / events_denom.weights.sum())

    # run model over (scaled) denominator event inputs
    trainer = L.Trainer(accelerator='gpu', devices=1)    
    X_denom = scaler.transform(events_denom.kinematics[features_4l].to_numpy())
    dl_denom = DataLoader(TensorDataset(torch.tensor(X_denom, dtype=torch.float32)), batch_size=1024)
    s_num_vs_denom = torch.cat(trainer.predict(dre, dl_denom))

    # likelihood ratio trick
    r_num_over_denom = s_num_vs_denom/(1-s_num_vs_denom)
    
    # inverse(!) of expectation value
    C_dre = 1.0/torch.sum(r_num_over_denom * p_denom)
    # ---

    return C_dre.numpy()

In [ ]:
C_sig_over_bkg = calibration_factor(model_sig_over_bkg, scaler_sig_over_bkg, events_bkg_train)
C_sbi_over_bkg = calibration_factor(model_sbi_over_bkg, scaler_sbi_over_bkg, events_bkg_train)
print(f'C(S/B)   = {C_sig_over_bkg:.6f}')
print(f'C(SBI/B) = {C_sbi_over_bkg:.6f}')

The inverse of the expectation value can be used as a *calibration factor* for the DRE as:

$$
\hat{r}_{\rm calib}(x;\mu_1,\mu_0) = C \times \hat{r}(x;\mu_1,\mu_0)
$$

You will need to keep those numbers for later use! 

---
**Questions for participants**

1. What limitations do you notice with this calibration mehtod?
1. Can this procedure be done with either $\mu_0$ and $\mu_1$? If so, what do you expect for each? If not, why?
---

## 2. Closure of the weights

A well-callibrated DRE it should provide a reweighting of the probability between hypotheses. 

$$
 \hat{p}(x;\theta_1) = p(x,z;\theta_0) \, \hat{r}(x ; \theta_1, \theta_0) 
$$

This time, let's perform this check using the *validation* data to evaluate the performance of the DNN independently of the training. The procedure is as follows:

1. Evaluate the calibrated DRE using the observables in the validation dataset.
1. Multiply DRE by the probaility of each event under $\mu_0$. 
1. Plot the original denominator hypothesis event distribution w.r.t. the $m_{4\ell}$ observable.
1. Plot the reweighted denominator-to-numerator hypothesis event distribution.
    - How significant of a shape change is this compared to #3?
1. Plot the distribution obtained directly from the numerator hypothesis.
    - Is the distribution #4 compatible, at least visually, with the correct answer?

In [ ]:
def reweighting_diagnostic(model, scaler, events_num_val, events_denom_val):
    # scale features
    X_denom_val = scaler.transform(events_denom_val.kinematics[features_4l].to_numpy())
    dl_denom_val = DataLoader(TensorDataset(torch.tensor(X_denom_val, dtype=torch.float32)), batch_size=1024) 
    
    # likelihood ratio trick
    trainer = L.Trainer(accelerator='gpu', devices=1)
    s_num_vs_denom = torch.cat(trainer.predict(model, dl_denom_val))
    r_num_over_denom = s_num_vs_denom / (1 - s_num_vs_denom)
    
    # histogram definition
    m4l_axis = hist.axis.Regular(41, 180, 1000, label = 'm4l')
    h_m4l_num = hist.Hist(m4l_axis)
    h_m4l_denom = hist.Hist(m4l_axis)
    h_m4l_num_from_denom = hist.Hist(m4l_axis)
    
    # observable calculation
    def calculate_m4l(kinematics):
        p_l1 = vector.array({'pt': kinematics['l1_pt'], 'eta': kinematics['l1_eta'], 'phi': kinematics['l1_phi'], 'energy': kinematics['l1_energy']})
        p_l2 = vector.array({'pt': kinematics['l2_pt'], 'eta': kinematics['l2_eta'], 'phi': kinematics['l2_phi'], 'energy': kinematics['l2_energy']})
        p_l3 = vector.array({'pt': kinematics['l3_pt'], 'eta': kinematics['l3_eta'], 'phi': kinematics['l3_phi'], 'energy': kinematics['l3_energy']})
        p_l4 = vector.array({'pt': kinematics['l4_pt'], 'eta': kinematics['l4_eta'], 'phi': kinematics['l4_phi'], 'energy': kinematics['l4_energy']})
        return (p_l1 + p_l2 + p_l3 + p_l4).mass
    
    # weights
    w_denom = torch.tensor(events_denom_val.weights)
    w_num = torch.tensor(events_num_val.weights)
    
    w_num_from_denom = torch.tensor(events_denom_val.weights) * r_num_over_denom
    
    # convert weights to probablities
    p_denom = w_denom / torch.sum(w_denom)
    p_num = w_num / torch.sum(w_num)
    p_num_from_denom = w_num_from_denom / torch.sum(w_num_from_denom)
    
    # populate each histogram with correct (obs, w)
    h_m4l_denom.fill( calculate_m4l(events_denom_val.kinematics), weight=p_denom)
    h_m4l_num.fill( calculate_m4l(events_num_val.kinematics), weight=p_num)
    h_m4l_num_from_denom.fill( calculate_m4l(events_denom_val.kinematics), weight=p_num_from_denom)
    
    # plotting
    fig, ax = plt.subplots(figsize=(6, 5))
    h_m4l_denom.plot(ax=ax, label=r'$p(x;\mu_0)$')
    h_m4l_num.plot(ax=ax, label=r'$p(x;\mu_1)$')
    h_m4l_num_from_denom.plot(ax=ax, label=r'$\frac{p(x;\mu_1)}{p(x;\mu_0)}\, p(x;\mu_0)$')
    ax.set_xlabel(r'$m_{4\ell}$', fontsize=12)
    ax.set_yscale('log')
    ax.legend()
    
reweighting_diagnostic(model_sbi_over_bkg, scaler_sbi_over_bkg, events_sbi_val, events_bkg_val)
reweighting_diagnostic(model_sig_over_bkg, scaler_sig_over_bkg, events_sig_val, events_bkg_val)

----
**Exercise**:

Modify the above function to explore how the distrbitions change by varying the binning ranges and widths. Try modifying the axis scale. 

Drawing from the above example, use the cell below to plot $m_{4\ell}$ for all three distributions. Try modifying the binning and the axis scaling

---

**Question for participants**
1. What interesting features did yuo find in your modifications?
---

## 3. Calibration curve

In section 2, we checked for kinematic closure on $m_{4\ell}$. In principle, this process is extendable to any set of kinematics; observable or latent. This means there is an infinite number of kinematics that can be checked with an infinite casting of the number of bins and bin boundaries. While this test can be useful in testing if latent kinematics should be made observable, it is impractical to use as a test for how well your DRE is capturing the true likelihood ratio.  

Recall that our neural networks are estimating the latent-aware classifier between hypotheses $\mu+0$ and $\mu_1$ using only observables. 

$$
    \hat s(x;\mu_1, \mu_0) \approx \frac{p(x,z;\mu_1)}{p(x,z;\mu_0) + p(x,z;\mu_1)}
$$

This means that the learned classifier will approximate the conditional expectation value of the target, latent-aware classifier. 

$$
    \hat{s}(x;\mu_1, \mu_0)\simeq\mathbb{E}[{s}(x,z;\theta_0, \mu_0)\mid x]
$$

One way to evaluate this expected behavior is through a counting test. We will:

1. Bin the true classifier function $s(x,z;\mu_1, \mu_0)$ and the learned (and calibrated) estimate of the classifier $\hat s(x;\mu_1, \mu_0)$ based on $\hat s(x;\mu_1, \mu_0)$
1. Weight the histograms by the latent-aware event weights $w(x,z;\theta_0$ and $w(x,z;\theta_1$
1. Plot these estimates against each other

If the DRE is well learned and calibrated, points will lie on the $x=y$ line (within variance). 

### 3.(a) Running the model over the numerator & denominator hypothesis datasets

We will make use of the `BalancedDataset` implementation to perform this test. 

In [ ]:
def calibration_curve(model, scaler, events_num, events_denom, calibration_factor):
    """
    Obtain calibration curve, C(x) of a DRE
    Parameters
    ---
    model: DRE model
    scaler: feature scaler
    events_num: numerator hypothesis events
    events_denom: denominator hypothesis events
    calibration_factor: overall calibration factor from expectation value
    """

    # classifier decision
    ds_balanced = BalancedDataset(events_num, events_denom, features=features_4l, scaler=scaler)
    dl = torch.utils.data.DataLoader(torch.tensor(ds_balanced.X, dtype=torch.float32), batch_size=1024)
    trainer = L.Trainer(accelerator='gpu', devices=1)
    s_num_vs_denom = torch.cat(trainer.predict(model, dl)).detach().numpy()

    # likelihood ratio trick -> apply calibration factor -> (lhrt)^(-1)
    r_num_over_denom = s_num_vs_denom/(1-s_num_vs_denom)
    r_num_over_denom * calibration_factor
    s_num_vs_denom = r_num_over_denom / (1+r_num_over_denom)
    # ---
    
    # (balanced) event weight?
    w_balanced = ds_balanced.w
    
    # true label (1 = numerator, 0 = denominator)
    y_num_or_denom = ds_balanced.s

    # s binning & histograms
    s_bins = np.linspace(0.0, 1.0, 41)
    s_centers = (s_bins[:-1] + s_bins[1:])/2
    s_nbins = len(s_bins) -1
    s_widths = (s_bins[1:] - s_bins[:-1])/2
    s_axis = hist.axis.Regular(40,0,1.0)
    h_num, h_denom = hist.Hist(s_axis, storage=hist.storage.Weight()), hist.Hist(s_axis, storage=hist.storage.Weight())

    # histogram using DRE as observable using balanced weights
    h_num.fill(s_num_vs_denom[y_num_or_denom==1], weight = w_balanced[y_num_or_denom==1])
    h_denom.fill(s_num_vs_denom[y_num_or_denom==0], weight = w_balanced[y_num_or_denom==0])
    
    N = h_num.values()
    D = h_denom.values()
    VN = h_num.variances()
    VD = h_denom.variances()
    s_mc_val = np.divide(N, N+D, out=np.zeros_like(N), where=(N+D)!=0)
    n_d_square = np.square(N + D)
    s_mc_err = np.sqrt(np.divide(D, n_d_square, out=np.zeros_like(D), where=n_d_square!=0)**2 * VN + np.divide(N, n_d_square, out=np.zeros_like(N), where=n_d_square!=0)**2 * VD)
    
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot([0,1], [0,1], marker='none', linestyle='--', color='grey')
    ax.set_xlabel(r'NSBI estimate, $\hat{s}(x)$', fontsize=12)
    ax.set_ylabel(r'MC estimate, $s(x)$', fontsize=12)
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    ax.errorbar(s_centers, s_mc_val, xerr=s_widths, yerr=s_mc_err, fmt='.', markersize=5, capsize=3, linewidth=1)

calibration_curve(model_sig_over_bkg, scaler_sig_over_bkg, events_sig_train, events_bkg_train, C_sig_over_bkg)
calibration_curve(model_sbi_over_bkg, scaler_sbi_over_bkg, events_sbi_train, events_bkg_train, C_sbi_over_bkg)

---
**Questions for participants**

1. Using the above plots, how would you characterize the calibration performance?
2. What do you notice about the range & variance of the curves between the two DREs? Why do you think such differences come about?

_Bonus question_: 

3. Why did we calculate an MC estimate of the true density ratio, $s(x,z)$? Why not compare these distributions at the event level?
---
